# Progress Review I — PCA Dimensionality Reduction

**Name:** *<your name>*  ·  **IT Number:** IT25103881  ·  **Group:** *<group ID>*
**Dataset:** UCI Letter Image Recognition (20,000 x 17)  ·  **Module:** IT2011 AIML

**Preprocessing activity:** PCA dimensionality reduction
**Outputs:** explained-variance plot (EDA visualization) + PCA projection visualization

Run with **Kernel -> Restart & Run All**. Paths resolve automatically (local or Google Colab).
Annotated walkthrough: `IT25103881_PCA_reference.ipynb`.

## 1. Technique

PCA finds orthogonal axes (principal components) that are linear combinations of the original
features, ordered by the variance each explains; keeping the first *k* reduces dimensionality while
retaining most of the variance. It is **unsupervised** (fitted on `X` only, never the label) and
**requires standardised input**, since it is variance-driven.

## 2. Setup and data loading

In [ ]:
# ── Imports and settings ──────────────────────────────────────────────────
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import joblib, zipfile
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# ── Project paths: works locally (notebook in notebooks/) and in Google Colab ──
DATA_FILE = "letter-recognition.data"

def locate_project_root():
    """Return the folder that contains data/raw, searching the usual locations."""
    candidates = [Path(".."), Path(".")]
    if Path("/content").is_dir():                        # Google Colab
        candidates.append(Path("/content"))
        candidates += [p.parent.parent for p in Path("/content").glob("*/data/raw")]
    for c in candidates:
        if (c / "data" / "raw").is_dir():
            return c.resolve()
    (Path("data") / "raw").mkdir(parents=True, exist_ok=True)
    return Path(".").resolve()

ROOT    = locate_project_root()
RAW_DIR = ROOT / "data" / "raw"
FIG_DIR = ROOT / "results" / "eda_visualizations"
OUT_DIR = ROOT / "results" / "outputs"
for d in (RAW_DIR, FIG_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

In [ ]:
# ── Locate the dataset: use the extracted file, else unzip the UCI archive ──
def get_data_path():
    direct = RAW_DIR / DATA_FILE
    if direct.exists():
        return direct
    for zp in [*RAW_DIR.glob("*.zip"), *(ROOT / "data").glob("*.zip"), *ROOT.glob("*.zip")]:
        with zipfile.ZipFile(zp) as z:
            z.extractall(RAW_DIR)
        print(f"Extracted {zp.name} into {RAW_DIR}")
        if direct.exists():
            return direct
    try:                                                  # fresh Colab session
        from google.colab import files
        print("Upload letter+recognition.zip (or letter-recognition.data):")
        for name in files.upload():
            src = Path(name)
            if src.suffix == ".zip":
                with zipfile.ZipFile(src) as z:
                    z.extractall(RAW_DIR)
            else:
                src.replace(RAW_DIR / src.name)
    except ImportError:
        raise FileNotFoundError(f"{DATA_FILE} not found in {RAW_DIR}")
    return RAW_DIR / DATA_FILE

RAW = get_data_path()
print("Using dataset:", RAW)

In [ ]:
# ── Load the dataset (headerless CSV; names from letter-recognition.names) ──
COLUMNS = ["letter",                                     # target: A-Z
           "x-box", "y-box", "width", "high", "onpix",   # box geometry + ink
           "x-bar", "y-bar",                             # first moments
           "x2bar", "y2bar", "xybar", "x2ybr", "xy2br",  # second moments
           "x-ege", "xegvy", "y-ege", "yegvx"]           # edge counts

df    = pd.read_csv(RAW, header=None, names=COLUMNS)
X_all = df.drop(columns="letter")     # 16 numeric features; the label never enters PCA
y_all = df["letter"]
print("Shape:", df.shape)
df.head()

## 3. PCA input requirements

In [ ]:
# ── PCA input requirements ────────────────────────────────────────────────
# PCA requires an all-numeric matrix with no missing values: a single NaN makes
# the decomposition fail. Verified here before fitting.'
print("Feature matrix :", X_all.shape)
print("Non-numeric    :", (X_all.dtypes == "object").sum())
print("Missing values :", int(X_all.isna().sum().sum()))
print("-> requirements met: PCA can be computed on this data as provided.")

## 4. Why PCA is needed for this dataset

Redundancy is quantified as a summary statistic below.

In [ ]:
# ── Why PCA is needed: quantify feature redundancy ───────────────────────
# PCA only pays off if the original features overlap in the information they
# carry. Summarised as two numbers rather than a full correlation matrix.
corr  = X_all.corr()
pairs = [(a, b, corr.loc[a, b])
         for i, a in enumerate(corr.columns) for b in corr.columns[i+1:]
         if abs(corr.loc[a, b]) > 0.5]
top   = max(pairs, key=lambda t: abs(t[2]))

print(f"Feature pairs with |r| > 0.5 : {len(pairs)} of 120")
print(f"Strongest pair               : {top[0]} <-> {top[1]}  r = {top[2]:+.3f}")
print("-> the 16 features are redundant, so PCA can compress them into fewer orthogonal axes.")

**15 of 120 feature pairs exceed |r| = 0.5**, the strongest being `x-box` ↔ `width` at **r = 0.852**.
The box-geometry features are redundant, so the 16 features do not carry 16 independent dimensions —
which is exactly what PCA exploits.

## 5. Prerequisites

Split before fitting anything, so no test information leaks into the transform, then standardise —
PCA requires standardised input.

In [ ]:
# ── Prerequisites for PCA: split first (no leakage), then standardise ─────
# PCA is variance-driven, so it requires standardised input: without it the
# components follow whichever feature happens to have the widest scale.
X_train, X_test = X_all.iloc[:16000], X_all.iloc[16000:]   # protocol from the .names file
y_train, y_test = y_all.iloc[:16000], y_all.iloc[16000:]

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on TRAIN only
X_test_scaled  = scaler.transform(X_test)        # reuse on TEST

print("Train:", X_train.shape, " Test:", X_test.shape)

## 6. Variance spectrum and choice of *k*

In [ ]:
# ── Fit PCA with all 16 components to obtain the variance spectrum ────────
pca_full = PCA(n_components=None, random_state=RANDOM_STATE).fit(X_train_scaled)
evr = pca_full.explained_variance_ratio_
cum = np.cumsum(evr)
eig = pca_full.explained_variance_          # eigenvalues, for Kaiser's rule

variance_table = pd.DataFrame({
    "Component":    [f"PC{i}" for i in range(1, len(evr) + 1)],
    "Eigenvalue":   eig.round(3),
    "Explained %":  (evr * 100).round(2),
    "Cumulative %": (cum * 100).round(2)})
display(variance_table)

In [ ]:
# ── FIGURE 1: explained-variance plot (the required EDA visualization) ────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
comps = np.arange(1, len(evr) + 1)

axes[0].bar(comps, evr * 100, color="steelblue", edgecolor="black", alpha=.85)
axes[0].axhline(100/16, ls="--", c="grey", lw=1, label="equal-share baseline (6.25%)")
axes[0].set(xlabel="Principal component", ylabel="Explained variance (%)",
            title="Variance explained per component (scree plot)", xticks=comps)
axes[0].legend()

axes[1].plot(comps, cum * 100, "o-", color="darkorange", lw=2, ms=6)
for thr, colour in [(85, "green"), (90, "purple"), (95, "red")]:
    axes[1].axhline(thr, ls="--", c=colour, lw=1, alpha=.7)
    k_thr = int(np.searchsorted(cum, thr/100) + 1)
    axes[1].annotate(f"{thr}% -> {k_thr} PCs", xy=(k_thr, thr),
                     xytext=(k_thr + .4, thr - 6), color=colour, fontsize=9)
axes[1].set(xlabel="Number of components", ylabel="Cumulative explained variance (%)",
            title="Cumulative explained variance", xticks=comps, ylim=(20, 103))

plt.tight_layout()
plt.savefig(FIG_DIR / "pca_explained_variance.png", dpi=150, bbox_inches="tight")
plt.show()

for t in (0.85, 0.90, 0.95):
    print(f"Cumulative variance >= {t:.0%} : {int(np.searchsorted(cum, t) + 1):2d} components")
print(f"Kaiser criterion (eigenvalue > 1): {int((eig > 1).sum()):2d} components")

**k = 12 selected** by the 95% cumulative-variance criterion (95.87% retained, 25% fewer dimensions).
Kaiser's rule (eigenvalue > 1) gives 5, retaining only 69.1% — too aggressive. PC1 explains only
26.81%, so the variance is unusually flat and this data cannot be compressed dramatically.

In [ ]:
# ── Apply PCA with the chosen k and project both splits ───────────────────
N_COMPONENTS = 12                       # 95% cumulative-variance criterion

pca         = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)   # fit on train
X_test_pca  = pca.transform(X_test_scaled)        # same rotation applied to test

print(f"16 features -> {N_COMPONENTS} components "
      f"({pca.explained_variance_ratio_.sum()*100:.2f}% of variance retained, "
      f"{(1 - N_COMPONENTS/16)*100:.0f}% reduction)")

pc_names     = [f"PC{i}" for i in range(1, N_COMPONENTS + 1)]
train_pca_df = pd.DataFrame(X_train_pca, columns=pc_names, index=X_train.index)
test_pca_df  = pd.DataFrame(X_test_pca,  columns=pc_names, index=X_test.index)

# The components are orthogonal, so the redundancy measured earlier is gone.
off_diag = train_pca_df.corr().values[~np.eye(N_COMPONENTS, dtype=bool)]
print(f"Max |correlation| between components: {np.abs(off_diag).max():.2e}  (~0)")

train_pca_df.head()

## 7. Component meaning and PCA visualization

In [ ]:
# ── FIGURE 2: loadings - what each component actually measures ────────────
loadings = pd.DataFrame(pca.components_.T, index=X_all.columns, columns=pc_names)

plt.figure(figsize=(12, 7))
sns.heatmap(loadings.iloc[:, :6], annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, linewidths=.5)
plt.title("PCA loadings - weight of each original feature in PC1-PC6")
plt.ylabel("Original feature")
plt.tight_layout()
plt.savefig(FIG_DIR / "pca_loadings.png", dpi=150, bbox_inches="tight")
plt.show()

for pc in ["PC1", "PC2", "PC3"]:
    idx = int(pc[2:]) - 1
    top = loadings[pc].abs().sort_values(ascending=False).head(4)
    print(f"{pc} ({pca.explained_variance_ratio_[idx]*100:.1f}% variance): "
          + ", ".join(f"{f} {loadings.loc[f, pc]:+.3f}" for f in top.index))

In [ ]:
# ── FIGURE 3: PCA visualization - letters projected onto PC1 vs PC2 ───────
SUBSET = ["A", "I", "M", "O", "W", "Z"]        # a readable subset of the 26 classes
mask   = y_train.isin(SUBSET).values

plt.figure(figsize=(9, 7))
sns.scatterplot(x=X_train_pca[mask, 0], y=X_train_pca[mask, 1], hue=y_train[mask].values,
                palette="tab10", s=18, alpha=.65, edgecolor="none")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%) - glyph size / ink")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%) - vertical mass")
plt.title("Letters projected onto the first two principal components")
plt.legend(title="Letter", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FIG_DIR / "pca_projection_pc1_pc2.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"PC1 + PC2 explain only {cum[1]*100:.1f}% of the variance.")

Classes form visible but heavily overlapping clouds — `I` at low PC1 (narrow, little ink), `M`/`W` at
high PC1 (wide, ink-heavy), matching PC1's loadings. PC1 + PC2 are only **43.3%** of the variance,
which is why 12 components are kept rather than 2.

## 8. Validation

In [ ]:
# ── Validation: how much class information survives the reduction ─────────
# KNN (k=5) is a probe to justify the choice of k, not the project's model.
results = []
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_scaled, y_train)
results.append(("All 16 features (baseline)", 16, 100.0,
                accuracy_score(y_test, knn.predict(X_test_scaled)) * 100))

for k in [12, 10, 7, 5, 2]:
    p   = PCA(n_components=k, random_state=RANDOM_STATE).fit(X_train_scaled)
    m   = KNeighborsClassifier(n_neighbors=5).fit(p.transform(X_train_scaled), y_train)
    acc = accuracy_score(y_test, m.predict(p.transform(X_test_scaled))) * 100
    results.append((f"{k} principal components", k, p.explained_variance_ratio_.sum()*100, acc))

comparison = pd.DataFrame(results, columns=["Feature set", "Dimensions",
                                            "Variance retained %", "Accuracy %"]).round(2)
comparison["Accuracy drop"] = (comparison.loc[0, "Accuracy %"] - comparison["Accuracy %"]).round(2)
display(comparison)

In [ ]:
# ── FIGURE 4: the accuracy / dimensionality trade-off behind k = 12 ───────
plot_df = comparison[comparison["Dimensions"] < 16].sort_values("Dimensions")

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(plot_df["Dimensions"], plot_df["Accuracy %"], "o-",
         color="crimson", lw=2, ms=7, label="KNN accuracy")
ax1.axhline(comparison.loc[0, "Accuracy %"], ls="--", c="grey", lw=1.2,
            label=f"all 16 features ({comparison.loc[0,'Accuracy %']:.2f}%)")
ax1.axvline(N_COMPONENTS, ls=":", c="navy", lw=1.5, label=f"selected k = {N_COMPONENTS}")
ax1.set_xlabel("Number of principal components retained")
ax1.set_ylabel("Test accuracy (%)", color="crimson")
ax1.tick_params(axis="y", labelcolor="crimson")

ax2 = ax1.twinx()
ax2.plot(plot_df["Dimensions"], plot_df["Variance retained %"], "s--",
         color="seagreen", lw=1.5, ms=6, alpha=.8, label="variance retained")
ax2.set_ylabel("Variance retained (%)", color="seagreen")
ax2.tick_params(axis="y", labelcolor="seagreen")
ax2.grid(False)

plt.title("Why k = 12: accuracy and variance vs. number of components")
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "pca_component_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()

12 components cost **0.83 percentage points** (94.65% → 93.82%) for 25% fewer dimensions and zero
inter-feature correlation. PCA is unsupervised, so no accuracy gain is expected; the benefits are
decorrelation, compression and interpretable axes.

## 9. Save outputs

In [ ]:
# ── Save the reduced dataset and the fitted PCA transformer ───────────────
train_out = train_pca_df.copy(); train_out["letter"] = y_train.values
test_out  = test_pca_df.copy();  test_out["letter"]  = y_test.values
train_out.to_csv(OUT_DIR / "letter_pca12_train.csv", index=False)
test_out.to_csv(OUT_DIR  / "letter_pca12_test.csv",  index=False)

# Fitted PCA, so the group pipeline reuses this exact rotation instead of re-fitting.
joblib.dump(pca, OUT_DIR / "pca_12_components.pkl")
variance_table.to_csv(OUT_DIR / "pca_explained_variance.csv", index=False)

print("results/outputs/ :", ", ".join(sorted(f.name for f in OUT_DIR.iterdir() if f.is_file())))
print("figures          :", ", ".join(sorted(f.name for f in FIG_DIR.iterdir() if f.suffix == ".png")))